In [4]:
# Cell 1: setup - sign in and connect to BigQuery
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
PROJECT_ID = "claude-project-511805"          # same ID as in your modelling notebook
client = bigquery.Client(project=PROJECT_ID)
print("Connected to", PROJECT_ID)

Connected to claude-project-511805


In [5]:
import json
import numpy as np
import pandas as pd

wk = client.query(
    "SELECT * FROM `ga4_funnel.weekly_metrics` ORDER BY week_start").to_dataframe()
wk["week_start"] = pd.to_datetime(wk["week_start"])

METRICS = {  # column -> (readable label, unit)
    "sessions":                ("Sessions",                 "count"),
    "revenue":                 ("Revenue",                  "usd"),
    "conv_rate_pct":           ("Session conversion rate",  "pct"),
    "view_to_checkout_pct":    ("View-to-checkout rate",    "pct"),
    "checkout_to_payment_pct": ("Checkout-to-payment rate", "pct"),
    "payment_to_purchase_pct": ("Payment-to-purchase rate", "pct"),
    "revenue_per_order":       ("Revenue per order",        "usd"),
}
CALENDAR = {
    "2020-11-23": "Thanksgiving / Black Friday week",
    "2020-11-30": "Cyber Monday week",
    "2020-12-21": "Christmas week (after typical gift-shipping cutoffs)",
    "2020-12-28": "New Year week (post-holiday period)",
}

def build_facts(i):
    row, prev = wk.iloc[i], wk.iloc[i - 1]
    past = wk.iloc[:i]                     # only weeks BEFORE this one (no peeking ahead)
    hist = past.tail(4)                    # trailing 4 weeks for the unusual-check
    week = row.week_start.strftime("%Y-%m-%d")
    facts = {"week_start": week,
             "calendar_note": CALENDAR.get(week),
             "metrics": [],
             "data_quality_warnings": []}

    for col, (label, unit) in METRICS.items():
        cur, before = float(row[col]), float(prev[col])
        change = cur - before
        z = None
        if len(hist) >= 3 and hist[col].std() > 0:
            z = (cur - hist[col].mean()) / hist[col].std()
        facts["metrics"].append({
            "metric":     label,
            "unit":       unit,
            "this_week":  round(cur, 2),
            "last_week":  round(before, 2),
            "change":     round(change, 2),   # percentage points for 'pct' metrics
            "pct_change": round(change / before * 100, 1) if before else None,
            "direction":  "up" if change > 0 else "down" if change < 0 else "flat",
            "unusual":    bool(z is not None and abs(z) >= 2),
        })

    typical_rpo = past["revenue_per_order"].median()
    if row.revenue_per_order < 0.5 * typical_rpo or row.revenue_per_order > 2 * typical_rpo:
        facts["data_quality_warnings"].append(
            f"Revenue per order ({row.revenue_per_order:.2f} USD) is far from the typical "
            f"{typical_rpo:.2f} USD while order volume looks normal; revenue tracking "
            f"may be incomplete this week. Do not interpret revenue changes as real.")
    return facts

all_facts = [build_facts(i) for i in range(1, len(wk))]
print("Weeks with facts:", len(all_facts))
print(json.dumps(all_facts[-1], indent=2))     # the last week, our tricky test case

Weeks with facts: 12
{
  "week_start": "2021-01-25",
  "calendar_note": null,
  "metrics": [
    {
      "metric": "Sessions",
      "unit": "count",
      "this_week": 26355.0,
      "last_week": 27484.0,
      "change": -1129.0,
      "pct_change": -4.1,
      "direction": "down",
      "unusual": false
    },
    {
      "metric": "Revenue",
      "unit": "usd",
      "this_week": 5546.0,
      "last_week": 26598.0,
      "change": -21052.0,
      "pct_change": -79.1,
      "direction": "down",
      "unusual": false
    },
    {
      "metric": "Session conversion rate",
      "unit": "pct",
      "this_week": 1.18,
      "last_week": 1.36,
      "change": -0.18,
      "pct_change": -13.2,
      "direction": "down",
      "unusual": false
    },
    {
      "metric": "View-to-checkout rate",
      "unit": "pct",
      "this_week": 9.4,
      "last_week": 10.5,
      "change": -1.1,
      "pct_change": -10.5,
      "direction": "down",
      "unusual": false
    },
    {
      "metr

In [7]:
!pip install -q google-genai

from google import genai
from google.genai import types
from google.colab import userdata

llm = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))   # 'llm', so it doesn't clash with BigQuery 'client'
MODEL = "gemini-3.8-flash"
# If MODEL errors, list what's available and pick the current "flash" model:
# for m in llm.models.list(): print(m.name)

SYSTEM = """You write a short weekly performance briefing for e-commerce managers.

Rules:
1. Use ONLY numbers that appear in the FACTS JSON. Never calculate new numbers. You may round to one decimal place.
2. Metrics with unit "pct" are rates. State levels with % (e.g. 1.18%). State their changes ONLY in percentage points using the "change" field, with before -> after (e.g. "down 0.18 percentage points, 1.36% -> 1.18%").
3. Metrics with unit "count" or "usd" may use "pct_change" as a % change.
4. If "data_quality_warnings" is not empty, the Headline must be about the data issue, and you must not describe the affected numbers as real business changes.
5. If "calendar_note" is present, mention it as context. Say changes "coincide with" it, never that it caused them.
6. Do not speculate about causes that are not in FACTS.
7. Prioritise metrics with "unusual": true, then conversion rate and revenue.
8. Output exactly this format, maximum 120 words, no bold or markdown:
Headline: <one sentence>
- <bullet>
- <bullet>
- <bullet>
Watch: <one sentence>"""

def generate_summary(facts, extra_instruction=""):
    prompt = "FACTS:\n" + json.dumps(facts, indent=2) + extra_instruction
    resp = llm.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(system_instruction=SYSTEM, temperature=0.2),
    )
    return resp.text.strip()

facts_by_week = {f["week_start"]: f for f in all_facts}
for week in ["2020-12-07", "2020-12-28", "2021-01-25"]:     # peak week, holiday dip, broken-data week
    print("=" * 12, week)
    print(generate_summary(facts_by_week[week]))

============ 2020-12-07
Headline: Revenue and sessions saw unusual growth alongside notable increases across checkout funnel rates.
- Revenue rose 39.4% to 59865 usd from 42948 usd, and Sessions increased 24.6% to 38739 from 31098.
- Checkout-to-payment rate was up 5.6 percentage points, 58.3% -> 63.9%.
- Payment-to-purchase rate was up 4.8 percentage points, 71.5% -> 76.3%.
Watch: Monitor Session conversion rate, which was up 0.09 percentage points, 1.9% -> 1.99%.
============ 2020-12-28
Headline: Overall performance drops coincided with the New Year week (post-holiday period).
- Session conversion rate saw an unusual drop, down 0.38 percentage points, 1.05% -> 0.67%.
- Revenue fell 30.2% to 10267 usd from 14716 usd.
- Sessions decreased 7.5% to 20446 from 22094.
Watch: Track whether session conversion rate recovers in the coming week.
============ 2021-01-25


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [8]:
import re, time

# Whole numbers for counts and big USD values, so the AI writes $59,865 rather than 59865.0
for f in all_facts:
    for m in f["metrics"]:
        if m["unit"] == "count" or (m["unit"] == "usd" and abs(m["this_week"]) >= 1000):
            for k in ["this_week", "last_week", "change"]:
                m[k] = int(round(m[k]))

SYSTEM += """
9. Format money as $12,345 (or $17.78 for values under 100) and counts as 12,345. Never write "usd".
10. If there is a data quality warning, describe the issue as possible, using "may" or "possible". Never state it as certain."""

In [9]:
UP   = ["up", "rose", "increase", "increased", "grew", "gain", "gains", "higher", "improved", "rebound", "rebounded"]
DOWN = ["down", "fell", "decrease", "decreased", "decline", "declined", "drop", "dropped", "lower", "fall"]

def allowed_numbers(facts):
    vals = set()
    for m in facts["metrics"]:
        for k in ["this_week", "last_week", "change", "pct_change"]:
            if m.get(k) is not None:
                vals.update({float(m[k]), abs(float(m[k]))})
    return vals

def number_ok(n, allowed):
    # accept rounding to 1 decimal, plus tiny slack for large values
    return any(abs(n - v) <= max(0.051, abs(v) * 0.001) for v in allowed)

def validate(text, facts):
    errors = []
    lines = [l.strip() for l in text.strip().splitlines() if l.strip()]

    # 1. Format
    if not lines or not lines[0].startswith("Headline:"):
        errors.append("First line must start with 'Headline:'.")
    if sum(l.startswith("- ") for l in lines) != 3:
        errors.append("There must be exactly 3 bullet lines starting with '- '.")
    if not lines or not lines[-1].startswith("Watch:"):
        errors.append("Last line must start with 'Watch:'.")
    if len(text.split()) > 130:
        errors.append(f"Too long: {len(text.split())} words (max 120).")

    # 2. Every number must exist in FACTS
    allowed = allowed_numbers(facts)
    for raw in re.findall(r"\d[\d,]*(?:\.\d+)?", text):
        if not number_ok(float(raw.replace(",", "")), allowed):
            errors.append(f"Number {raw} does not appear in FACTS.")

    # 3. Direction words must match the real direction
    for line in lines:
        low = line.lower()
        for m in facts["metrics"]:
            if m["metric"].lower() in low:
                said_up   = any(re.search(rf"\b{w}\b", low) for w in UP)
                said_down = any(re.search(rf"\b{w}\b", low) for w in DOWN)
                if m["direction"] == "up" and said_down and not said_up:
                    errors.append(f"'{m['metric']}' went up, but the text says down.")
                if m["direction"] == "down" and said_up and not said_down:
                    errors.append(f"'{m['metric']}' went down, but the text says up.")

    # 4. Data-quality weeks: lead with it, and hedge it
    if facts["data_quality_warnings"]:
        if not any(w in lines[0].lower() for w in ["data", "tracking"]):
            errors.append("Headline must lead with the data-quality issue.")
        if not re.search(r"\b(may|possible|possibly|likely)\b", text.lower()):
            errors.append("Describe the data issue as possible ('may'/'possible'), not certain.")
    return errors

# Test the checker on a deliberately WRONG summary before trusting it
bad = ("Headline: Revenue rose 45% this week.\n- Session conversion rate increased.\n"
       "- Sessions dropped.\n- Something else.\nWatch: Keep an eye on it.")
print(validate(bad, facts_by_week["2020-12-28"]))

['Number 45 does not appear in FACTS.', "'Revenue' went down, but the text says up.", "'Session conversion rate' went down, but the text says up."]


In [10]:
def fallback_summary(facts):
    m = {x["metric"]: x for x in facts["metrics"]}
    c, r, s = m["Session conversion rate"], m["Revenue"], m["Sessions"]
    head = ("Headline: Possible data tracking issue this week; treat revenue figures with caution."
            if facts["data_quality_warnings"]
            else f"Headline: Session conversion rate was {c['this_week']}% this week.")
    return "\n".join([
        head,
        f"- Session conversion rate: {c['last_week']}% -> {c['this_week']}% ({c['change']:+} percentage points).",
        f"- Revenue: ${r['last_week']:,} -> ${r['this_week']:,} ({r['pct_change']:+}%).",
        f"- Sessions: {s['last_week']:,} -> {s['this_week']:,} ({s['pct_change']:+}%).",
        "Watch: See the dashboard for step-level detail."])

def briefing(facts, max_retries=2):
    time.sleep(30)                                  # stay under the free-tier rate limit
    text = generate_summary(facts)
    for attempt in range(max_retries + 1):
        errors = validate(text, facts)
        if not errors:
            return {"week_start": facts["week_start"], "summary": text,
                    "status": "passed", "attempts": attempt + 1, "errors": []}
        if attempt < max_retries:
            feedback = ("\n\nYour previous draft failed these checks. Rewrite it and fix ALL of them:\n- "
                        + "\n- ".join(errors) + "\n\nPrevious draft:\n" + text)
            time.sleep(5)
            text = generate_summary(facts, feedback)
    return {"week_start": facts["week_start"], "summary": fallback_summary(facts),
            "status": "fallback", "attempts": max_retries + 1, "errors": errors}

for week in ["2020-12-07", "2020-12-28", "2021-01-25"]:
    out = briefing(facts_by_week[week])
    print("=" * 12, week, "|", out["status"], "| attempts:", out["attempts"])
    print(out["summary"])
    if out["errors"]:
        print("Last errors:", out["errors"])

============ 2020-12-07 | passed | attempts: 1
Headline: Revenue reached $59,865 alongside unusual increases in sessions and checkout conversion rates.
- Revenue rose 39.4% to $59,865 while sessions grew 24.6% to 38,739.
- Checkout-to-payment rate was up 5.6 percentage points, 58.3% -> 63.9%, and payment-to-purchase rate was up 4.8 percentage points, 71.5% -> 76.3%.
- Session conversion rate was up 0.09 percentage points, 1.9% -> 1.99%.
Watch: Monitor checkout-to-payment rate and payment-to-purchase rate to see if these elevated rates continue next week.
============ 2020-12-28 | passed | attempts: 1
Headline: Performance decreases coincide with the New Year week (post-holiday period), led by an unusual drop in session conversion rate.
- Session conversion rate was down 0.38 percentage points, 1.05% -> 0.67%.
- Revenue was down 30.2% from $14,716 to $10,267.
- Sessions were down 7.5% from 22,094 to 20,446, while revenue per order rose 18.7% from $63.16 to $74.94.
Watch: Monitor session

In [11]:
import re

# keep the original validator; the guard makes re-running this cell safe
if "_validate_base" not in globals():
    _validate_base = validate

def validate(text, facts):
    errors = _validate_base(text, facts)
    flagged = {m["metric"].lower() for m in facts["metrics"] if m["unusual"]}
    for line in text.splitlines():
        if "unusual" not in line.lower():
            continue
        named = [m["metric"].lower() for m in facts["metrics"]
                 if m["metric"].lower() in line.lower()]
        if not named:
            errors.append(f"Says 'unusual' but names no metric: {line[:60]}")
        elif not any(n in flagged for n in named):
            errors.append(f"Calls {named} unusual, but FACTS does not flag it.")
    return errors

# 1) re-check the 3 real summaries with the new rule
for wk_ in ["2020-12-07", "2020-12-28", "2021-01-25"]:
    r = briefing(facts_by_week[wk_])
    print(wk_, r["status"], r["attempts"], r["errors"])

# 2) force the model to always return the bad summary and check the fallback kicks in
real_gen = generate_summary
generate_summary = lambda facts, extra_instruction="": bad
forced = briefing(facts_by_week["2020-12-28"])
generate_summary = real_gen          # always restore the real one
print("\nFORCED:", forced["status"], "| attempts:", forced["attempts"])
print(forced["summary"])

2020-12-07 passed 1 []
2020-12-28 passed 1 []
2021-01-25 passed 1 []

FORCED: fallback | attempts: 3
Headline: Session conversion rate was 0.67% this week.
- Session conversion rate: 1.05% -> 0.67% (-0.38 percentage points).
- Revenue: $14,716 -> $10,267 (-30.2%).
- Sessions: 22,094 -> 20,446 (-7.5%).
Watch: See the dashboard for step-level detail.


In [14]:
MODEL = "gemini-3.5-flash-lite"

In [15]:
import os, json, pickle
CACHE = "/content/eval_cache.pkl"
cache = pickle.load(open(CACHE, "rb")) if os.path.exists(CACHE) else {}

def cached(key, fn):
    if key not in cache:
        cache[key] = fn()
        pickle.dump(cache, open(CACHE, "wb"))   # saved after every call
    return cache[key]

In [16]:
import time, pandas as pd

NAIVE_PROMPT = ("Write a short weekly performance summary for a manager. "
                "Format: a 'Headline:' line, 3 bullets starting with '- ', and a 'Watch:' line.\n\nFACTS:\n")

def naive_summary(facts):                      # same facts and format, but no rules and no validator
    time.sleep(10)
    r = llm.models.generate_content(model=MODEL, contents=NAIVE_PROMPT + json.dumps(facts, indent=2))
    return r.text.strip()

def bucket(err):
    e = err.lower()
    if "number" in e and "facts" in e: return "number"
    if "but the text says" in e:       return "direction"
    if "unusual" in e:                 return "unusual"
    return "format_or_dq"

rows = []
for wk_, f in sorted(facts_by_week.items()):
    n_text = cached(("naive", wk_), lambda: naive_summary(f))
    g = cached(("guarded", wk_), lambda: briefing(f))
    for name, text, status, attempts in [("naive", n_text, "-", 1),
                                         ("guarded", g["summary"], g["status"], g["attempts"])]:
        errs = validate(text, f)
        counts = {b: 0 for b in ["number", "direction", "unusual", "format_or_dq"]}
        for e in errs: counts[bucket(e)] += 1
        rows.append({"week_start": wk_, "pipeline": name, "status": status, "attempts": attempts,
                     **counts, "summary": text, "errors": " | ".join(errs)})
    print(wk_, "| naive errors:", len(validate(n_text, f)), "| guarded:", g["status"], g["attempts"])

ev = pd.DataFrame(rows)
ev["factual"] = ev[["number", "direction", "unusual"]].sum(axis=1)
print(ev.groupby("pipeline")[["number", "direction", "unusual", "format_or_dq", "factual"]].sum())
print("\nWeeks with ≥1 factual error:",
      ev.assign(bad=ev.factual > 0).groupby("pipeline")["bad"].sum().to_dict())
print("Guarded status:", ev[ev.pipeline == "guarded"].status.value_counts().to_dict())

2020-11-09 | naive errors: 2 | guarded: fallback 3
2020-11-16 | naive errors: 1 | guarded: passed 1
2020-11-23 | naive errors: 2 | guarded: passed 1
2020-11-30 | naive errors: 0 | guarded: passed 2
2020-12-07 | naive errors: 1 | guarded: passed 2
2020-12-14 | naive errors: 0 | guarded: passed 1
2020-12-21 | naive errors: 1 | guarded: passed 1
2020-12-28 | naive errors: 0 | guarded: passed 1
2021-01-04 | naive errors: 2 | guarded: passed 1
2021-01-11 | naive errors: 3 | guarded: passed 1
2021-01-18 | naive errors: 1 | guarded: passed 1
2021-01-25 | naive errors: 4 | guarded: passed 1
          number  direction  unusual  format_or_dq  factual
pipeline                                                   
guarded        0          0        0             0        0
naive          7          8        1             1       16

Weeks with ≥1 factual error: {'guarded': 0, 'naive': 9}
Guarded status: {'passed': 11, 'fallback': 1}


In [17]:
import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

sh = gc.create("ga4_weekly_briefings")
out = ev[ev.pipeline == "guarded"][["week_start", "status", "attempts", "summary"]]
sh.sheet1.update([out.columns.tolist()] + out.astype(str).values.tolist())
ws2 = sh.add_worksheet("evaluation", rows=40, cols=12)
full = ev.drop(columns=["summary"])
ws2.update([full.columns.tolist()] + full.astype(str).values.tolist())
print(sh.url)

https://docs.google.com/spreadsheets/d/1VPGPS8XVQhU4NeESbqypoQ7_s6dW_osDUOCY0YgEz2w


In [18]:
DATE_RE = re.compile(
    r"\b\d{4}-\d{2}-\d{2}\b"
    r"|\b(?:oct|nov|dec|jan|feb)[a-z]*\.?\s+\d{1,2}(?:st|nd|rd|th)?(?:,?\s+\d{4})?\b"
    r"|\b\d{1,2}(?:st|nd|rd|th)?\s+(?:oct|nov|dec|jan|feb)[a-z]*\.?(?:\s+\d{4})?\b", re.I)

def validate(text, facts):
    clean = DATE_RE.sub("[date]", text)          # dates are not metrics
    errors = _validate_base(clean, facts)
    flagged = {m["metric"].lower() for m in facts["metrics"] if m["unusual"]}
    for line in clean.splitlines():
        if "unusual" not in line.lower():
            continue
        named = [m["metric"].lower() for m in facts["metrics"] if m["metric"].lower() in line.lower()]
        if not named:
            errors.append(f"Says 'unusual' but names no metric: {line[:60]}")
        elif not any(n in flagged for n in named):
            errors.append(f"Calls {named} unusual, but FACTS does not flag it.")
    return errors

for wk_ in sorted(facts_by_week):
    text = cache[("naive", wk_)]
    errs = validate(text, facts_by_week[wk_])
    if errs:
        print(f"===== {wk_} | {errs}\n{text}\n")

===== 2020-11-09 | ['Number 5 does not appear in FACTS.']
Headline: Revenue rose 8.3% to $23,624 for the week of Nov 9 despite a 9.9% drop in sessions, driven by a surge in conversion rate.
- Session conversion rate jumped 51.0% week-over-week to reach 1.51%.
- View-to-checkout and checkout-to-payment rates improved significantly by 16.9% and 14.4%, respectively.
- Revenue per order declined 20.5% to $69.69 compared to $87.63 last week.
Watch: Monitor whether the drop in traffic (sessions) continues to be offset by higher conversion rates moving forward.

===== 2020-11-23 | ['Number 2 does not appear in FACTS.', "'Revenue' went up, but the text says down."]
Headline: Strong Thanksgiving and Black Friday week drove a 25.5% surge in revenue alongside improved conversion rates.
- Sessions grew 7.2% to 27,788, while overall session conversion rate jumped 27.6% to 1.85%.
- Strong funnel progression showed significant gains in view-to-checkout (19.6%) and payment-to-purchase (71.6%) rates.
-

In [19]:
cache.pop(("guarded", "2020-11-09"), None)
g = cached(("guarded", "2020-11-09"), lambda: briefing(facts_by_week["2020-11-09"]))
print(g["status"], g["attempts"], g["errors"]); print(g["summary"])

fallback 3 ["'Revenue' went up, but the text says down."]
Headline: Session conversion rate was 1.51% this week.
- Session conversion rate: 1.0% -> 1.51% (+0.51 percentage points).
- Revenue: $21,820 -> $23,624 (+8.3%).
- Sessions: 24,996 -> 22,519 (-9.9%).
Watch: See the dashboard for step-level detail.


In [35]:
DATE_RE = re.compile(
    r"\b\d{4}-\d{2}-\d{2}\b"
    r"|\b(?:oct(?:ober)?|nov(?:ember)?|dec(?:ember)?|jan(?:uary)?|feb(?:ruary)?)\.?\s+\d{1,2}(?:st|nd|rd|th)?\b(?:,?\s+\d{4}\b)?"
    r"|\b\d{1,2}(?:st|nd|rd|th)?\s+(?:oct(?:ober)?|nov(?:ember)?|dec(?:ember)?|jan(?:uary)?|feb(?:ruary)?)\b(?:\s+\d{4}\b)?", re.I)
UP3   = UP + ["surged", "surge", "surges", "jumped", "jump", "doubled", "climbed", "rise", "rises", "growth", "grow"]
DOWN3 = DOWN + ["dipped", "dip", "dips", "slipped", "declines", "drops", "falls", "decreases"]
CAUSAL = re.compile(r"\b(driven by|due to|because|caused by|reflecting|impact\w*|bolstered by|supported by|thanks to|as a result|led to|leading to|capitaliz\w*)\b", re.I)
LEAK   = re.compile(r"\b(facts|do not interpret|should not be interpreted)\b", re.I)
CLAUSE = re.compile(r",\s|;| while | but | despite | even as | whereas | and ", re.I)

def named_metrics(s, facts):          # longest label first, so "revenue per order" isn't also read as "revenue"
    found, rest = [], s.lower()
    for m in sorted(facts["metrics"], key=lambda m: -len(m["metric"])):
        lab = m["metric"].lower()
        if lab in rest:
            found.append(m); rest = rest.replace(lab, " ")
    return found

def validate_v3(text, facts):
    errors = []
    clean = DATE_RE.sub("[date]", text)
    lines = [l.strip() for l in clean.strip().splitlines() if l.strip()]
    if not lines or not lines[0].startswith("Headline:"): errors.append("First line must start with 'Headline:'.")
    if sum(l.startswith("- ") for l in lines) != 3:      errors.append("There must be exactly 3 bullet lines.")
    if not lines or not lines[-1].startswith("Watch:"):  errors.append("Last line must start with 'Watch:'.")
    if len(text.split()) > 130:                          errors.append(f"Too long: {len(text.split())} words.")
    allowed = allowed_numbers(facts)
    for raw in re.findall(r"\d[\d,]*(?:\.\d+)?", clean):
        if not number_ok(float(raw.replace(",", "")), allowed):
            errors.append(f"Number {raw} does not appear in FACTS.")
    flagged = {m["metric"] for m in facts["metrics"] if m["unusual"]}
    for line in lines:
        low = line.lower()
        is_watch = line.startswith("Watch:")
        if (c := CAUSAL.search(low)): errors.append(f"Claims a cause ('{c.group(0)}'); rule 7 forbids it.")
        if (k := LEAK.search(low)):   errors.append(f"Copies internal instruction text ('{k.group(0)}').")
        if re.search(r"\busd\b", low): errors.append("Write money as $1,234, never 'usd'.")
        for clause in CLAUSE.split(low):
            up   = any(re.search(rf"\b{w}\b", clause) for w in UP3)
            down = any(re.search(rf"\b{w}\b", clause) for w in DOWN3)
            for m in named_metrics(clause, facts):
                if not is_watch and m["direction"] == "up" and down and not up:
                    errors.append(f"'{m['metric']}' went up, but the text says down.")
                if not is_watch and m["direction"] == "down" and up and not down:
                    errors.append(f"'{m['metric']}' went down, but the text says up.")
                if m["unit"] in ("pct", "%", "percent") and m["pct_change"] is not None and "percentage point" not in clause:
                    for raw in re.findall(r"(\d+(?:\.\d+)?)\s*%", clause):
                        v = float(raw)
                        if abs(v - abs(m["pct_change"])) <= 0.051 and all(abs(v - x) > 0.051 for x in (m["this_week"], m["last_week"])):
                            errors.append(f"'{m['metric']}' change given as relative % ({raw}%); use percentage points.")
        if "unusual" in low:
            ms = named_metrics(low, facts)
            if not ms: errors.append(f"Says 'unusual' but names no metric: {line[:60]}")
            elif not any(m["metric"] in flagged for m in ms): errors.append(f"Calls {[m['metric'] for m in ms]} unusual, but FACTS does not flag it.")
    if facts["data_quality_warnings"]:
        if not any(w in lines[0].lower() for w in ["data", "tracking"]): errors.append("Headline must lead with the data-quality issue.")
        if not re.search(r"\b(may|possible|possibly|likely)\b", text.lower()): errors.append("Describe the data issue as possible, not certain.")
    return errors

validate = validate_v3          # briefing() uses the new checker from now on

In [36]:
def kind(e):
    for k, key in [("cause", "Claims a cause"), ("leak", "Copies"), ("relative_pct", "relative %"),
                   ("number", "does not appear"), ("direction", "but the text says"),
                   ("unusual", "unusual"), ("data_issue", "data")]:
        if key in e: return k
    return "format"

rows = []
for wk_ in sorted(facts_by_week):
    for pipe in ["naive", "guarded"]:
        res = cache[(pipe, wk_)]
        text = res if pipe == "naive" else res["summary"]
        for e in validate_v3(text, facts_by_week[wk_]):
            rows.append({"week": wk_, "pipeline": pipe, "kind": kind(e), "error": e})
rs = pd.DataFrame(rows)
print(rs.pivot_table(index="kind", columns="pipeline", values="week", aggfunc="count", fill_value=0))
print("\nWeeks with ≥1 error:", rs.groupby("pipeline").week.nunique().to_dict(), "of 12")
pd.set_option("display.max_colwidth", 110); print(rs.to_string(index=False))

pipeline      naive
kind               
cause            19
data_issue        1
relative_pct      8
unusual           1

Weeks with ≥1 error: {'naive': 11} of 12
      week pipeline         kind                                                                                            error
2020-11-09    naive        cause                                                 Claims a cause ('driven by'); rule 7 forbids it.
2020-11-09    naive relative_pct             'Session conversion rate' change given as relative % (51.0%); use percentage points.
2020-11-16    naive        cause                                                 Claims a cause ('driven by'); rule 7 forbids it.
2020-11-16    naive        cause                                              Claims a cause ('supported by'); rule 7 forbids it.
2020-11-16    naive        cause                                                  Claims a cause ('impacted'); rule 7 forbids it.
2020-11-16    naive relative_pct               'View-to-ch

In [29]:
cache.pop(("guarded", "2021-01-25"), None)
g = cached(("guarded", "2021-01-25"), lambda: briefing(facts_by_week["2021-01-25"]))
print(g["status"], g["attempts"], g["errors"]); print(g["summary"])

passed 2 []
Headline: Revenue tracking may be incomplete this week as revenue per order is $17.78 compared to the typical $73.81 while order volume looks normal.
- Revenue per order is down 75.0%, $71.12 -> $17.78
- Revenue is down 79.1%, $26,598 -> $5,546
- Session conversion rate is down 0.18 percentage points, 1.36% -> 1.18%
Watch: Monitor tracking tags and data feeds for potential issues affecting order values.


In [37]:
out = pd.DataFrame([{"week_start": w, **{k: cache[("guarded", w)][k] for k in ["status", "attempts", "summary"]}}
                    for w in sorted(facts_by_week)])
sh.sheet1.clear(); sh.sheet1.update([out.columns.tolist()] + out.astype(str).values.tolist())
ws2.clear();       ws2.update([rs.columns.tolist()] + rs.astype(str).values.tolist())

{'spreadsheetId': '1VPGPS8XVQhU4NeESbqypoQ7_s6dW_osDUOCY0YgEz2w',
 'updatedRange': 'evaluation!A1:D30',
 'updatedRows': 30,
 'updatedColumns': 4,
 'updatedCells': 120}